# 喜马拉雅 Colab Worker（GitHub 版）

代码从 GitHub 自动拉取，无需手动上传。更新代码后只需在 GitHub push，重新运行此 notebook 即可使用最新版本。

## 特性
- 自动从 GitHub 克隆最新代码
- 支持 `--num-workers` 多线程并行处理
- tqdm 实时进度条
- 所有配置通过 VPS 全局设置下发（TG Token / Cookie / 代理 / 降噪模型等）

> **使用方法**：修改下方配置后，从上到下依次执行每个 Cell。

## 1. 配置

修改下方参数后执行：

In [ ]:
# ═══ VPS 连接 ═══
VPS_URL = "http://117.55.234.219:59388"
WORKER_ID = "colab_mt_001"        # 留空则自动生成
WORKER_TOKEN = "gchqRbJR2kmop5YwQQeYNGLs"

# ═══ 多线程 ═══
import os
NUM_WORKERS = os.cpu_count() or 2  # 根据 CPU 核心数自动设置

# ═══ GitHub 仓库 ═══
GIT_REPO = "https://github.com/collinsgraciano/ximalaya_manager.git"
GIT_BRANCH = "main"               # 分支名

# ═══ 运行参数 ═══
POLL_INTERVAL = 10                # 无任务时等待秒数
MAX_JOBS = 0                      # 最大处理任务数 (0=不限)

print(f"配置完成 | CPU: {os.cpu_count()} 核 | 线程数: {NUM_WORKERS}")

## 2. 克隆代码 & 安装依赖

每次运行都从 GitHub 拉取最新代码。

In [ ]:
!rm -rf /content/ximalaya_manager
!git clone -b {GIT_BRANCH} {GIT_REPO} /content/ximalaya_manager

# 安装依赖
!pip install -q requests pycryptodome tqdm pydub
!apt-get -qq install -y ffmpeg

print("代码克隆 & 依赖安装完成")

## 3. 启动 Worker

执行后开始轮询任务。代码直接从 GitHub 克隆的 `.py` 文件运行，无需内嵌代码。

In [ ]:
import os, subprocess, sys

worker_id = WORKER_ID or f"colab_mt_{os.urandom(4).hex()}"

cmd = [
    sys.executable, "/content/ximalaya_manager/colab/ximalaya_colab_worker.py",
    "--vps-url", VPS_URL,
    "--worker-id", worker_id,
    "--worker-token", WORKER_TOKEN,
    "--num-workers", str(NUM_WORKERS),
    "--poll-interval", str(POLL_INTERVAL),
    "--max-jobs", str(MAX_JOBS),
    "--install-deps",
]
print(f"启动: {' '.join(cmd)}", flush=True)
os.execvp(sys.executable, cmd)